# MLflow: experiment tracking for ML models

**Goal:** Compare different ML models with MLflow experiment tracking

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

# Set MLflow tracking URI to SQLite database
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("ML Classification Comparison")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment: {mlflow.get_experiment_by_name('ML Classification Comparison').name}")

In [ ]:
# Load data
df = pd.read_csv('synthetic_classification_dataset.csv')
print(f"Dataset shape: {df.shape}")
print("\nFirst 3 rows:")
print(df.head(3))

In [ ]:
print("DATA OVERVIEW")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Number of classes: {df['target'].nunique()}")
print(f"Class distribution: {df['target'].value_counts().to_dict()}")
print("\nDescriptive statistics")
print(df.describe().round(2))

In [ ]:
print("CORRELATION WITH TARGET")
correlations = df.drop('target', axis=1).corrwith(df['target']).sort_values(key=abs, ascending=False)
for feature, corr in correlations.items():
    print(f"{feature}: {corr:.3f}")

# Visualize correlations
plt.figure(figsize=(8, 4))
correlations.plot(kind='bar')
plt.title('Feature Correlation with Target')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Data preparation
X = df.drop('target', axis=1)
y = df['target']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Train class distribution: {y_train.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}")

In [ ]:
# Check scaling necessity
print("FEATURE SCALE ANALYSIS")
feature_stats = df.drop('target', axis=1).describe().transpose()

max_mean = feature_stats['mean'].max()
min_mean = feature_stats['mean'].min()
scale_diff = (max_mean - min_mean) / min_mean * 100

print(f"Max mean: {max_mean:.2f}")
print(f"Min mean: {min_mean:.2f}")
print(f"Scale difference: {scale_diff:.1f}%")

if scale_diff > 50:
    print("SCALING REQUIRED")
else:
    print("Scaling not required")

def train_and_evaluate_with_mlflow(model, X_train, X_test, y_train, y_test, model_name, use_scaled=False):
    """Train model and log to MLflow"""
    
    # Determine which data to use
    train_data = X_train_scaled if use_scaled else X_train
    test_data = X_test_scaled if use_scaled else X_test
    
    with mlflow.start_run(run_name=model_name) as run:
        # Log parameters
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("scaling", use_scaled)
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("test_size", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_classes", len(np.unique(y_train)))
        
        # Log model-specific hyperparameters
        if hasattr(model, 'get_params'):
            params = model.get_params()
            for param_name, param_value in params.items():
                if param_value is not None and isinstance(param_value, (int, float, str, bool)):
                    mlflow.log_param(f"model_{param_name}", param_value)
        
        # Training
        model.fit(train_data, y_train)
        
        # Predictions
        y_train_pred = model.predict(train_data)
        y_test_pred = model.predict(test_data)
        
        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_f1 = f1_score(y_test, y_test_pred, average='weighted')
        overfitting = train_acc - test_acc
        
        # Log metrics
        mlflow.log_metric("train_accuracy", train_acc)
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_f1_score", test_f1)
        mlflow.log_metric("overfitting", overfitting)
        
        # Log confusion matrix as artifact
        cm = confusion_matrix(y_test, y_test_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - {model_name}')
        plt.ylabel('True')
        plt.xlabel('Predicted')
        plt.savefig('confusion_matrix.png')
        plt.close()
        mlflow.log_artifact('confusion_matrix.png')
        
        # Log model
        signature = infer_signature(train_data, model.predict(train_data))
        mlflow.sklearn.log_model(model, "model", signature=signature)
        
        # Log dataset info
        dataset_info = {
            'train_shape': train_data.shape,
            'test_shape': test_data.shape,
            'feature_names': list(X_train.columns)
        }
        mlflow.log_dict(dataset_info, "dataset_info.json")
        
        print(f"  Train Acc: {train_acc:.3f}, Test Acc: {test_acc:.3f}, F1: {test_f1:.3f}")
        
        metrics = {
            'model': model_name,
            'train_acc': train_acc,
            'test_acc': test_acc,
            'test_f1': test_f1,
            'overfitting': overfitting,
            'run_id': run.info.run_id
        }
        
        return metrics, model, run.info.run_id

In [ ]:
# Train models with MLflow logging
results = []
models = {}
run_ids = {}

print("TRAINING MODELS WITH MLflow LOGGING")

# Logistic Regression
print("\nLogistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000)
metrics, model, run_id = train_and_evaluate_with_mlflow(
    lr, X_train, X_test, y_train, y_test, "Logistic Regression", use_scaled=True
)
results.append(metrics)
models['lr'] = model
run_ids['lr'] = run_id

# KNN
print("\nKNN...")
knn = KNeighborsClassifier(n_neighbors=5)
metrics, model, run_id = train_and_evaluate_with_mlflow(
    knn, X_train, X_test, y_train, y_test, "KNN", use_scaled=True
)
results.append(metrics)
models['knn'] = model
run_ids['knn'] = run_id

# SVM
print("\nSVM...")
svm = SVC(random_state=42, probability=True)
metrics, model, run_id = train_and_evaluate_with_mlflow(
    svm, X_train, X_test, y_train, y_test, "SVM", use_scaled=True
)
results.append(metrics)
models['svm'] = model
run_ids['svm'] = run_id

# Decision Tree
print("\nDecision Tree...")
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
metrics, model, run_id = train_and_evaluate_with_mlflow(
    dt, X_train, X_test, y_train, y_test, "Decision Tree", use_scaled=False
)
results.append(metrics)
models['dt'] = model
run_ids['dt'] = run_id

# Naive Bayes
print("\nNaive Bayes...")
nb = GaussianNB()
metrics, model, run_id = train_and_evaluate_with_mlflow(
    nb, X_train, X_test, y_train, y_test, "Naive Bayes", use_scaled=False
)
results.append(metrics)
models['nb'] = model
run_ids['nb'] = run_id

print("\nAll models trained and logged to MLflow!")

In [ ]:
# Compare results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('test_f1', ascending=False)

print("MODEL COMPARISON")
display_df = results_df.drop('run_id', axis=1).round(3)
print(display_df.to_string(index=False))

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
results_df.plot(kind='bar', x='model', y=['train_acc', 'test_acc'], ax=axes[0])
axes[0].set_title('Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].legend(['Train', 'Test'])
axes[0].tick_params(axis='x', rotation=45)

# F1-score
results_df.plot(kind='bar', x='model', y='test_f1', ax=axes[1], color='green')
axes[1].set_title('F1-Score (Test)')
axes[1].tick_params(axis='x', rotation=45)

# Overfitting
results_df.plot(kind='bar', x='model', y='overfitting', ax=axes[2], color='red')
axes[2].set_title('Overfitting')
axes[2].axhline(y=0.1, color='black', linestyle='--', alpha=0.5)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Save comparison to MLflow
with mlflow.start_run(run_name="Model_Comparison") as run:
    mlflow.log_dict(display_df.to_dict(), "model_comparison.json")
    
    # Create comparison plot
    plt.figure(figsize=(12, 6))
    results_df_melted = results_df.melt(id_vars=['model'], 
                                        value_vars=['train_acc', 'test_acc', 'test_f1'],
                                        var_name='metric', value_name='value')
    sns.barplot(data=results_df_melted, x='model', y='value', hue='metric')
    plt.title('Model Performance Comparison')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('model_comparison.png')
    plt.close()
    mlflow.log_artifact('model_comparison.png')

In [ ]:
# Overfitting analysis
print("OVERFITTING ANALYSIS")
overfit_threshold = 0.1

overfitting = results_df[results_df['overfitting'] > overfit_threshold]
stable = results_df[results_df['overfitting'] <= overfit_threshold]

print(f"Models with overfitting (> {overfit_threshold}):")
if len(overfitting) > 0:
    for _, row in overfitting.iterrows():
        print(f"  - {row['model']}: {row['overfitting']:.3f}")
else:
    print("  None")

print(f"\nStable models (<= {overfit_threshold}):")
for _, row in stable.iterrows():
    print(f"  - {row['model']}: {row['overfitting']:.3f}")

In [ ]:
# Top-3 models
print("TOP-3 MODELS")
top_3 = results_df.head(3)
for i, (_, row) in enumerate(top_3.iterrows(), 1):
    print(f"{i}. {row['model']} - F1: {row['test_f1']:.3f}, Acc: {row['test_acc']:.3f}")

# Best model
best_model_name = results_df.iloc[0]['model']
best_run_id = results_df.iloc[0]['run_id']
print(f"\nBest model: {best_model_name}")
print(f"MLflow Run ID: {best_run_id}")

# Detailed report for best model
print("\nDetailed report for best model")
model_mapping = {
    'Logistic Regression': (models['lr'], True),
    'KNN': (models['knn'], True),
    'SVM': (models['svm'], True),
    'Naive Bayes': (models['nb'], False),
    'Decision Tree': (models['dt'], False)
}

best_model, use_scaled = model_mapping.get(best_model_name)
if best_model:
    X_test_final = X_test_scaled if use_scaled else X_test
    y_pred = best_model.predict(X_test_final)
    print(classification_report(y_test, y_pred))

In [ ]:
# MLflow experiment summary
print("MLFLOW EXPERIMENT SUMMARY")
print(f"Experiment: {mlflow.get_experiment_by_name('ML Classification Comparison').name}")
print(f"Experiment ID: {mlflow.get_experiment_by_name('ML Classification Comparison').experiment_id}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

# Get all runs
experiment = mlflow.get_experiment_by_name("ML Classification Comparison")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

print(f"\nTotal runs: {len(runs)}")
print("\nRun details:")
for _, run in runs.iterrows():
    print(f"  - {run['tags.mlflow.runName']}: F1={run['metrics.test_f1_score']:.3f}, Acc={run['metrics.test_accuracy']:.3f}")

In [ ]:
# How to access MLflow UI
print("HOW TO ACCESS MLflow UI")
print("1. Start MLflow UI server:")
print("   mlflow ui")
print("\n2. Open browser and go to:")
print("   http://localhost:5000")
print("\n3. In MLflow UI you can:")
print("   - View experiment 'ML Classification Comparison'")
print("   - Compare model performance metrics")
print("   - View confusion matrices for each model")
print("   - Download trained models")
print("   - Track hyperparameters and artifacts")
print("\n4. To start MLflow server from this directory:")
print("   Open terminal in C:/Users/Ruslan/Desktop/0964")
print("   Run: mlflow ui")

# Save run information for easy access
run_info = {
    'experiment_name': 'ML Classification Comparison',
    'experiment_id': experiment.experiment_id,
    'tracking_uri': mlflow.get_tracking_uri(),
    'runs': results_df[['model', 'run_id', 'test_f1', 'test_acc']].to_dict('records')
}

import json
with open('mlflow_run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)

print("\nRun information saved to 'mlflow_run_info.json'")

In [ ]:
# Final conclusions
print("FINAL CONCLUSIONS")
print(f"1. Best model: {best_model_name}")
print(f"2. Quality (F1): {results_df.iloc[0]['test_f1']:.3f}")
print(f"3. Accuracy: {results_df.iloc[0]['test_acc']:.3f}")
print(f"4. Models with overfitting: {len(overfitting)} out of {len(results_df)}")
print(f"5. Scaling required for: Logistic Regression, KNN, SVM")
print(f"6. MLflow experiment: 'ML Classification Comparison'")

print("\nSUMMARY")
if len(overfitting) > 0:
    print("- For models with overfitting, regularization or complexity reduction is recommended")
if results_df.iloc[0]['test_f1'] > 0.8:
    print("- Model quality is high, can be used")
else:
    print("- Model quality is poor, additional hyperparameter tuning is recommended")

print("\nMLflow BENEFITS:")
print("- All experiments tracked and reproducible")
print("- Models saved and can be loaded for future use")
print("- Metrics and parameters logged for comparison")
print("- Confusion matrices and artifacts stored")
print("- Easy experiment comparison in UI")